In [ ]:
!date

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import sys
import time
import os
import pandas as pd


#from stardist.matching import group_matching_labels
from skimage.io import imread
import tifffile

from collections import defaultdict

# from scipy.spatial import cKDTree
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.colors import ListedColormap
plt.rcParams['figure.dpi']=200
plt.rcParams['savefig.dpi']=200

**necessary for processing nuclear masks and assigning dots**

In [ ]:
import kspaces.EM
import kspaces.affine_subspace_



**necessary for making heatmaps and scatterplots**

In [ ]:

import sklearn
import anndata


# functions to process masks

In [ ]:
def load_positions(file, skip_first_line = True, num_hyb_rounds = 20, cycle_length = 201):
    
    positions = np.load(file,allow_pickle = True)[int(skip_first_line):]
    
    #file is cyclical but we only need the first set of entries
    num_slots = int(len(positions)/num_hyb_rounds) 
    positions = positions[:num_slots]
    
    #trim off last column indicating hyb round
    positions = positions[:,:-1]
    
    #david annotated only the first X neurons. There are also gaps with nan for some reason.
    #trim to last non-nan entry
    last_entry = -1
    if len(np.where(~np.isnan(positions[:cycle_length,0]))[0]) != 0:
        last_entry = np.where(~np.isnan(positions[:cycle_length,0]))[0][-1]
    
    
    positions = positions[:last_entry+1]
    
    return positions

def get_mask_ID(n, nuclear_mask):
    name = n['name']
    try:
        ID =  nuclear_mask[int(n.z),int(n.x),int(n.y)]
        if name == '-':
            print(f'unnamed cell {name} at (z,x,y):{n.z,n.x,n.y}')
        return ID
    except:
        print(f'invalid coordinates for cell {name}:{n.z,n.x,n.y}')
        return -1
    
def combine_annotations_with_masks(pool, positions = [4,11]):
    
    neuron_names = pd.read_excel(f'POOL{pool}/MG1_Pool{pool}_POINTS-labels.xlsx')
    neuron_names['POSITION'] = neuron_names['POSITION'].apply(lambda x: f'pos{x}'.split('.')[0])

    for pos in positions:
        print()
        print('Pos',pos)
        #COMBINE ANNOTATION POSITIONS AND NAMES
        #try:
        names = neuron_names[neuron_names.POSITION == f'pos{pos}'].values[:,1:]
        coords = load_positions(f'POOL{pool}/Annotation_coordinates/Pos{pos}_points.npy')[:len(names)]
        input_locations = pd.DataFrame(np.concatenate([names,coords], axis = 1), columns = ['name','pos','x','y','z']) 
        input_locations = input_locations[['x','y','z','name']]
        #except:
        #    print(f'Prior annotations file POOL{pool}/Annotation_coordinates/Pos{pos}_points.npy not found')
        #    input_locations = pd.DataFrame(columns = ['name','pos','x','y','z'])
        
        #COMBINE WITH NUCLEAR MASK IDs
        #nuclear_mask = np.load(f'POOL{pool}/nuc_masks/pos{pos}/Masks_hyb_000_pos{pos}.npy',allow_pickle = True)
        nuclear_mask = np.load(f'POOL{pool}/nuc_masks/pos{pos}/Masks_hyb_0_pos{pos}.npy',allow_pickle = True)

        input_locations['maskID'] = input_locations.apply(lambda x: get_mask_ID(x, nuclear_mask), axis = 1)
        input_locations = input_locations[input_locations['maskID'] != -1]

        
        
        maskIDs = np.unique(nuclear_mask)
        if 0 in maskIDs:
            maskIDs = maskIDs[1:]
        
        print('Locating cells and finding centers. This will take a few minutes.')
        coords = np.zeros((len(maskIDs),3))
        
        for i, ID in enumerate(maskIDs):
            coords[i] = np.mean(np.argwhere(nuclear_mask == ID),axis = 0).astype(int)
        
        temp = maskIDs.reshape(len(maskIDs),1)
        cell_coordinates = pd.DataFrame(np.hstack([temp,temp,coords]), 
                                columns = ['maskID','name','z','x','y'])
        cell_coordinates['maskID'] = cell_coordinates['maskID'].astype(int)
        cell_coordinates['name'] = cell_coordinates['maskID'].apply(lambda x: cell_coordinates_helper(x, input_locations))
        cell_coordinates['notes'] = np.array([['']]*len(maskIDs))
        cell_coordinates.to_csv(f'POOL{pool}/output/annotation_coordinates_pos{pos}.csv',index = False)
        print()

def cell_coordinates_helper(x, input_locations):
    if x in input_locations['maskID'].values:
        r = input_locations[input_locations['maskID'] == x]['name'].values
        if len(r) > 1:
            print(f'Same neuron annotated twice: {r}. Using first annotation')
        return r[0]
        
    else:
        return x

# functions to process dots

In [ ]:

    
def make_distance_grid(search_radius):
    d = 2*search_radius+1
    x_dist = np.tile(np.arange(d),d).reshape(d,d) - (d-1)/2
    y_dist = x_dist.T
    return np.sqrt(x_dist**2 + y_dist**2)

def clip_grid(dot, x1, x2, y1, y2, distance_grid,radius):
    grid = distance_grid[max(x1- (dot[1]-radius),0):x2-(dot[1]+radius)+len(distance_grid), 
        max(y1- (dot[2]-radius),0):y2-(dot[2]+radius)+len(distance_grid)]
    return grid



def make_cellxgene_matrix_hard_assignment(pool,pos, genes,df):
    cells = []
    neuron_names = pd.read_csv(f'POOL{pool}/output/annotation_coordinates_pos{pos}.csv')[['name','maskID']]
    for ID in df.maskID.unique():
        if ID == 0 or ID == -1:
            continue
        slice_ = df[df.maskID == ID]
        name = ID
        if ID in neuron_names.maskID.values:
            if len(neuron_names[neuron_names.maskID == ID].name.values) > 1:
                print(f'warning: more than one neuron name ({neuron_names[neuron_names.maskID == ID].name.values}) assigned to mask ID {ID}')
            name = neuron_names[neuron_names.maskID == ID].name.values[0]
        cell = [name]
        cell.append(ID)
        for g in genes:
            cell.append(np.sum(slice_.gene == g))
        cells.append(cell)

    matrix = pd.DataFrame(cells, columns = np.concatenate([['cell','maskID'],genes]))
    matrix.to_csv(f'POOL{pool}/output/cellxgene_hard_assignment_pos{pos}.csv', index = False)
    


def make_cellxgene_matrix_soft_assignment(pool, pos, genes,maskIDs, df):
    #df['assignment'] = df['assignment'].apply(ast.literal_eval)
    matrix = np.zeros((len(maskIDs), len(genes)))
    matrix = pd.DataFrame(matrix, columns = genes)
    matrix['maskID'] = maskIDs
    matrix['idx'] = maskIDs
    matrix = matrix.set_index('idx')
    cell_names = pd.read_csv(f'POOL{pool}/output/annotation_coordinates_pos{pos}.csv')[['name','maskID']]#.set_index('maskID').to_dict().get('name')
    cell_names = cell_names.sort_values(by = 'maskID')
    matrix['cell'] = np.concatenate([[0], cell_names['name'].values])#matrix.maskID.apply(lambda x: cell_names.get(x, 'unannotated_cell'))
    df.apply(lambda x: cellxgene_helper(x, matrix), axis = 1)


    matrix =  matrix[ np.concatenate([['cell','maskID'],genes])]
    matrix.to_csv(f'POOL{pool}/output/cellxgene_soft_assignment_pos{pos}.csv', index = False)

def cellxgene_helper(x, matrix):
    for c in x.assignment:
        matrix.loc[c[0],x.gene] += c[1]
        

In [ ]:

def assign_prob_v3(dot, nearby_cells, cells):
    cells_ = [cells.get(c) for c in  nearby_cells]
    dot_ = dot[1:] # skip z coordinate
    probs = kspaces.EM.E_step(np.array([dot_,dot_]), cells_, assignment = 'soft')[0] 
    return list(zip(nearby_cells,probs))



In [ ]:


def make_cells(mask):
    cells = {}
    for z, layer in enumerate(mask):
        cells_ = {}

        for c in np.unique(layer):
            pixels = np.argwhere(layer == c)
            center = np.mean(pixels,axis = 0)
            
            if c!= 0:
                try:
                    cell = kspaces.EM.fit_single_space(np.argwhere(layer == c), 1)
                    #cell.latent_sigmas =  2* cell.latent_sigmas
                    #cell.sigma = 2*cell.sigma
                    #cell.prior = np.sqrt(cell.latent_sigmas[0]*cell.sigma)
                    #cells_[c] = cell
                    
                    pca = sklearn.decomposition.PCA(n_components = 2)
                    transformed_pixels = sklearn.decomposition.PCA(n_components = 2).fit_transform(pixels).T
                    long_axis = np.max(transformed_pixels[0]) - np.min(transformed_pixels[0])
                    short_axis = np.max(transformed_pixels[1]) - np.min(transformed_pixels[1])


                    
                    cell.latent_sigmas = [long_axis]
                    cell.sigma = short_axis

                    cell.prior = np.sqrt(long_axis*short_axis)
                    cells_[c] = cell
                except:
                    if pixels.shape[0] < 3: #only 1 or 2 pixels in z stack
                        long_axis = [[1,1]] #doesnt matter what direction
                        latent_sigma = 1
                        sigma = 1
                        prior = 1
                        cell = kspaces.affine_subspace_.affine_subspace(long_axis, 
                                                                        center,
                                                                        sigma,
                                                                        [latent_sigma],
                                                                        prior)
                        cells_[c] = cell
                    else:
                        raise RuntimeError(f'error on nuclear mask ID {c} in stack {z}. ')
        sum_priors = 0
        for c in cells_.values():
            sum_priors += c.prior
        for c in cells_.values():
            c.prior = c.prior/sum_priors

        cells[z] = cells_
    
    return cells


def assign_dots_v4(pool, positions, max_distance):
    genes = pd.read_excel(f'POOL{pool}/Hyb info.xlsx', header = 1).Gene.values    
    for pos in positions:
        print()
        print('Pos ',pos)
        print('-------')
        nuclear_mask = np.load(f'POOL{pool}/nuc_masks/pos{pos}/Masks_hyb_0_pos{pos}.npy',allow_pickle = True)
        cells = make_cells(nuclear_mask)
        dfs = []
        for i, row in pd.read_excel(f'POOL{pool}/Hyb info.xlsx', header = 1).iterrows():
            channel = row.Fluorophore
            gene = row.Gene
            hyb = row['Hyb round']
            try:
                dots = np.load(f'POOL{pool}/pos{pos}/spots_{channel}_hyb_{str(hyb).zfill(3)}_pos{pos}.npy', allow_pickle = True)
            except:
                try:
                    dots = np.load(f'POOL{pool}/pos{pos}/spots_{channel}_hyb_{str(hyb).zfill(2)}_pos{pos}.npy', allow_pickle = True)
                except:
                    dots = np.load(f'POOL{pool}/pos{pos}/spots_{channel}_hyb_{str(hyb)}_pos{pos}.npy', allow_pickle = True)
            dots = pd.DataFrame(dots, columns = ['z','x','y'])

            if len(dots) ==0:
                print(f'No dots for {gene}. hyb: {hyb} ch: {channel}')
                continue
            distance_grid = make_distance_grid(max_distance)
            dots['assignment'] = dots.apply(lambda x: assign_dot_helper_v4(x.values, 
                                                                    cells.get(x.z),
                                                                    nuclear_mask, 
                                                                    distance_grid,
                                                                    max_distance), 
                                        axis = 1)
            dots['maskID'] = dots['assignment'].apply(lambda x: x[np.argmax(np.array(x)[:,1])][0])
            dots['hyb'] = hyb
            dots['channel'] = channel
            dots['gene'] = gene
            dfs.append(dots)

        df = pd.concat(dfs)
        df = df.reset_index()
        df.to_csv(f'POOL{pool}/output/assigned_dots_pos{pos}.csv')
        print(f'{len(df)} dots total.')
        print(f'{np.sum(df.maskID == 0)} unassigned dots too far from a mask with max_distance = {max_distance}')
        print(f'{np.sum(df.maskID == -1)} unassigned dots equidistant from two or more nuclear masks')
        
        make_cellxgene_matrix_hard_assignment(pool, pos, genes,df)
        maskIDs = np.unique(nuclear_mask)
        #if 0 in maskIDs:
        #    maskIDs = maskIDs[1:]
        make_cellxgene_matrix_soft_assignment(pool, pos, genes,maskIDs,df)

In [ ]:
def assign_dot_helper_v4(dot, cells, nuclear_mask, distance_grid, radius):
    image_size = 2048
    z = dot[0]
    
    if nuclear_mask[dot[0],dot[1],dot[2]] != 0: #if inside a nuclear mask, hard assign it to that mask
        return [[nuclear_mask[dot[0],dot[1],dot[2]],1]]
    
    x1 = max(0, dot[1]-radius)
    x2 = min(dot[1]+radius,image_size-1)
    y1 = max(0, dot[2]-radius)
    y2 = min(dot[2]+radius,image_size-1)
    mask_patch = nuclear_mask[z, x1:x2+1, y1:y2+1]
    try:
        IDs = search_patch_v4(dot, mask_patch, clip_grid(dot, x1, x2, y1, y2, distance_grid,radius), radius, cells)
    except:
        IDs = [-2]
        print(f'error on dot {dot}')
        
    if len(IDs) == 0:
        return [(0,1)]
    if len(IDs) == 1:
        return [(IDs[0],1)]
 
    return assign_prob_v3(dot, IDs, cells)

def search_patch_v4(dot, mask_patch, distance_grid, max_distance, cells):
    nearby_cells = []        
    
   
    for n in np.unique(mask_patch):
        if n == 0:
            nearby_cells.append([n, np.inf])
        else:
            nearby_cells.append([n, np.min(distance_grid[mask_patch == n])])
                
    
    nearby_cells = np.array(nearby_cells)
    nearby_cells = nearby_cells[nearby_cells[:,1] <= max_distance] #square shaped grid
    
    
    if len(nearby_cells) == 0:
        return []
    
    #discard distances
    nearby_cells = nearby_cells[:,0]
    
    #determine for which of the cells the dot is within an ellipse double the size of the nuclear mask
    temp = []
    for c in nearby_cells:
        cell = cells.get(int(c))
        if cell.orthogonal_distance(dot[1:]) < cell.sigma: #2*short axis radius
             if np.absolute(cell.displacement(dot[1:])) < cell.latent_sigmas[0]: #2*long axis radius
                    temp.append(int(c))
    nearby_cells = np.array(temp)
                                
    if len(nearby_cells) == 0:
        return []
                                
    return nearby_cells

# Process masks and dots

POOL 1 example

In [ ]:
positions = [0,1,2,3,4,5,6,8,9,10,11]
pool = 1
max_distance = 15

In [ ]:
combine_annotations_with_masks(pool, positions)

In [ ]:
assign_dots_v4(pool, positions, max_distance)